#### simple self attention without trainable weights

In [1]:
import torch
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your     (x^1)
    [0.55, 0.87, 0.66], # journey  (x^2)
    [0.57, 0.85, 0.64], # starts   (x^3)
    [0.22, 0.58, 0.33], # with     
    [0.77, 0.25, 0.10], # one      
    [0.05, 0.80, 0.55]] # step     
)

In [2]:
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i,query)
print(attn_scores_2)


tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [3]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum() # normalisation
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In [4]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)
softmax_naive(attn_scores_2)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [5]:
torch.softmax(attn_scores_2, dim=0)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [6]:
0.1385+0.2379+0.2333+ 0.1240+0.1082+0.1581

1.0

In [7]:
attn_scores = torch.empty(6,6)
for i , x_i in enumerate(inputs):
    for j , x_j in enumerate(inputs):
        attn_scores[i,j] = torch.dot(x_i,x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [8]:
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores , dim=1)
print(attn_weights) # 6x6

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [9]:
torch.sum(torch.tensor([0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452]))

tensor(0.9999)

In [10]:
all_context_vecs = attn_weights @ inputs # 6x6 6x3 = 6 x 3
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

#### Implenting **Self Attention** with trainable weights

In [11]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [12]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_in
d_out = 2 # for showing purpose we are setting our own size , generally in and out are same size

torch.nn.Parameter is a special type of tensor in PyTorch that is used to represent learnable parameters of a neural network, such as weights and biases.

Not every tensor in a model should be trainable.
nn.Parameter explicitly tells PyTorch: this tensor should be optimized.

In [13]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out))  # .parameter makes the weights trainable makes it
W_key = torch.nn.Parameter(torch.rand(d_in, d_out))
W_value = torch.nn.Parameter(torch.rand(d_in, d_out))
W_query

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]], requires_grad=True)

In [14]:
query_2 = x_2 @ W_query #  1x3 3x2 = 1x2
query_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [15]:
keys = inputs @ W_key # 6x3 3x2
values = inputs @ W_value
keys.shape

torch.Size([6, 2])

In [16]:
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)

In [17]:
keys_2 = keys[1]
attn_scores_2 = torch.dot(query_2,keys_2)
attn_scores_2

tensor(1.8524, grad_fn=<DotBackward0>)

In [18]:
attn_scores_2 = query_2 @ keys.T
attn_scores_2 # you can see the 1.8524 here too 

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [19]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5 , dim = -1) # dim = -1 last dimension that is normalisation along columns or 
# you can say your weights of each word etc . good convention to follow 

In [20]:
print(attn_weights_2)
print(attn_weights_2.sum())

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)
tensor(1., grad_fn=<SumBackward0>)


In [21]:
context_vec_2 = attn_weights_2 @ values
context_vec_2


tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

##### Implementing a compact self attention

In [22]:
"""
def __init__(self, d_in, d_out): — Constructor that takes:

d_in = input embedding dimension (e.g., 3 in your notebook)
d_out = output dimension for queries/keys/values (e.g., 2)
super().__init__() — Calls the parent nn.Module's constructor. 
                    This initializes the module properly so PyTorch can manage
                    it. Essential line—always include it when inheriting from nn.Module.
"""
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in , d_out):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Parameter(torch.rand(d_in , d_out))
        self.W_key = nn.Parameter(torch.rand(d_in , d_out))
        self.W_value = nn.Parameter(torch.rand(d_in , d_out))

    def forward(self , x): # say if our x is 6x3 and weights be 3x2
        keys = x @ self.W_key       # 6x2 
        queries = x @ self.W_query   # 6x2 
        values = x @ self.W_value        # 6x2 
        attn_scores = queries @ keys.T # omega  6x2 2x6
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5 , dim = -1
        )
        context_vec = attn_weights @ values # 6x6 6x2
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in,d_out)
sa_v1(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [23]:
m = torch.nn.Linear(2,3)
print(m.bias)
print(m.weight)

Parameter containing:
tensor([-0.3189,  0.2240, -0.3146], requires_grad=True)
Parameter containing:
tensor([[-0.1668,  0.2270],
        [ 0.5000,  0.1317],
        [ 0.1934,  0.6825]], requires_grad=True)


In [24]:
import torch.nn as nn

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in , d_out , qkv_bias = False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in , d_out ,bias=qkv_bias)
        self.W_key = nn.Linear(d_in , d_out)
        self.W_value = nn.Linear(d_in , d_out)

    def forward(self , x): # say if our x is 6x3 and weights be 3x2
        keys = self.W_key(x)      # 6x2 
        queries = self.W_query(x)   # 6x2 
        values = self.W_value(x)        # 6x2 
        attn_scores = queries @ keys.T # omega  6x2 2x6
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5 , dim = -1
        )
        context_vec = attn_weights @ values # 6x6 6x2
        return context_vec

torch.manual_seed(123)  # if you change the seed number your outputs may vary from v1 and v2 cause of changing weights numbers
sa_v2 = SelfAttention_v2(d_in,d_out)
sa_v2(inputs)

tensor([[0.0127, 0.2929],
        [0.0146, 0.2897],
        [0.0146, 0.2897],
        [0.0167, 0.2888],
        [0.0153, 0.2903],
        [0.0167, 0.2885]], grad_fn=<MmBackward0>)

##### Applying casual attention mask

In [25]:
# Your journey starts with one step

In [26]:
keys = sa_v2.W_key(inputs)      # 6x2 
queries = sa_v2.W_query(inputs)   # 6x2 
values = sa_v2.W_value(inputs)        # 6x2 
attn_scores = queries @ keys.T # omega  6x2 2x6
attn_weights = torch.softmax(
    attn_scores / keys.shape[-1]**0.5 , dim = -1
)
attn_weights

tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)

In [27]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length,context_length)) # tril returns a diagonal lower triangular matrix
mask_simple

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [28]:
masked_simple = attn_weights * mask_simple
masked_simple

tensor([[0.1717, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1749, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1637, 0.1749, 0.1746, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.0000, 0.0000],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<MulBackward0>)

In [29]:
row_sums = masked_simple.sum(dim=-1 ,  keepdim=True) #keepdim=True means don’t drop the reduced dimension; keep it with size 1.
masked_simple_norm = masked_simple / row_sums
masked_simple_norm

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<DivBackward0>)

so instead of computing normalised scores during self attention and again then renormalising during masking what we do is
we computing attention scores and normalise during the time of masking only .

In [30]:
mask = torch.triu(torch.ones(context_length, context_length),diagonal=1)
masked = attn_scores.masked_fill(mask.bool() , -torch.inf)
print(masked)


tensor([[0.3454,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.3237, 0.4184,   -inf,   -inf,   -inf,   -inf],
        [0.3225, 0.4160, 0.4135,   -inf,   -inf,   -inf],
        [0.1516, 0.2086, 0.2070, 0.1649,   -inf,   -inf],
        [0.2116, 0.2575, 0.2563, 0.1688, 0.1821,   -inf],
        [0.1757, 0.2472, 0.2452, 0.2011, 0.1758, 0.2247]],
       grad_fn=<MaskedFillBackward0>)


In [31]:
torch.exp(torch.tensor(-99999999))

tensor(0.)

In [32]:
torch.exp(torch.tensor(float("-inf")))

tensor(0.)

In [33]:
attn_weights = torch.softmax(masked / d_k**0.5 , dim = 1)
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)

##### using dropout for masking
Suppose dropout probability = p

Only (1 − p) of neurons survive

To keep the expected activation unchanged, frameworks use inverted dropout

So surviving activations are scaled by:
            1/ (1-p)	​


In [34]:
torch.manual_seed(123)

layer = torch.nn.Dropout(0.5)

In [35]:
example = torch.ones(6,6)
example

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [36]:
layer(example) #  1/ 1-0.5 is 2 

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])

In [37]:
batch = torch.stack((inputs , inputs ) , dim=0)
batch # 2, 6, 3 ;: 2 represents batches 

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])

In [38]:
import torch.nn as nn

class CasualAttention(nn.Module):
    def __init__(self, d_in , d_out , context_length, dropout, qkv_bias = False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in , d_out ,bias=qkv_bias)
        self.W_key = nn.Linear(d_in , d_out)
        self.W_value = nn.Linear(d_in , d_out)
        self.dropout = torch.nn.Dropout(dropout)
        self.register_buffer("mask" , torch.triu(torch.ones(context_length, context_length), diagonal=1))

        """ When a model is moved to GPU using .to(device) or .cuda(), all parameters and registered buffers
            move automatically, but manually created tensors do NOT unless they are registered or explicitly moved.
            register_buffer is used so such tensors move with the model.
        """
    def forward(self ,x):
        batch_size , num_tokens , d_in = x.shape # x  = 2x6x3
        queries = self.W_query(x)       # (2,6,3) --> (2,6,2) similarly the below ones too
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1,2) # 3x2 and 3,2 cannot so 3x2 2x3 , here (2,6,2) aand (2,6,2)
                                                    #  so swap dimension 1 and 2  which is (6,2) and (2,6) = 6,6 
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens , :num_tokens] , -torch.inf
        ) # _ tells just that it is in place or same as x = x + 5 ( just an example )
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5 , dim = -1
        )
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ values # 6x6 6x2
        return context_vec

torch.manual_seed(123)  # if you change the seed number your outputs may vary from v1 and v2 cause of changing weights numbers

context_length = batch.shape[1]
ca = CasualAttention(d_in,d_out,context_length, dropout=0.5) # din = 3 and dout = 2
ca(batch)



tensor([[[-0.1838,  0.8293],
         [-0.0888,  0.4008],
         [-0.1205,  0.4863],
         [-0.0169,  0.4538],
         [-0.0403,  0.1316],
         [ 0.0715,  0.0974]],

        [[-0.1838,  0.8293],
         [-0.0888,  0.4008],
         [-0.0586,  0.2645],
         [-0.1409,  0.5310],
         [ 0.0890,  0.3819],
         [ 0.0134,  0.3850]]], grad_fn=<UnsafeViewBackward0>)

#### Extending single-head attention to multi-head attention

In [39]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self , d_in, d_out , context_length , dropout, num_heads = 2 , qkv_bias = False):
        super().__init__()
        self.heads = nn.ModuleList([
            CasualAttention(d_in , d_out , context_length , dropout , qkv_bias) for _ in range(num_heads)
        ])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads] , dim=-1)

torch.manual_seed(123)


In [40]:
batch.shape

torch.Size([2, 6, 3])

In [41]:
torch.manual_seed(123)
context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)


tensor([[[-0.0919,  0.4146, -0.6492,  0.2716],
         [-0.0949,  0.3654, -0.6891,  0.2708],
         [-0.0935,  0.3520, -0.7029,  0.2734],
         [-0.0333,  0.3082, -0.6288,  0.3013],
         [ 0.0057,  0.3238, -0.6055,  0.3448],
         [ 0.0167,  0.2885, -0.5782,  0.3368]],

        [[-0.0919,  0.4146, -0.6492,  0.2716],
         [-0.0949,  0.3654, -0.6891,  0.2708],
         [-0.0935,  0.3520, -0.7029,  0.2734],
         [-0.0333,  0.3082, -0.6288,  0.3013],
         [ 0.0057,  0.3238, -0.6055,  0.3448],
         [ 0.0167,  0.2885, -0.5782,  0.3368]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


Instead of maintaining two separate classes, `MultiHeadAttentionWrapper` and
`CausalAttention`, we can combine both of these concepts into a single
`MultiHeadAttention` class. 

Also, in addition to just merging the
`MultiHeadAttentionWrapper` with the `CausalAttention` code, we will make some other
modifications to implement multi-head attention more efficiently

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in , d_out ,context_length , dropout,num_heads , qkv_bias = False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        """      
            # d_in = 3
            # d_out = 2
            # num_heads = 2
            # head_dim = 1
        """
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in , d_out , bias = qkv_bias)
        self.W_key = nn.Linear(d_in , d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out , d_out)  # Use a Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask' , torch.triu(torch.ones(context_length , context_length) , diagonal = 1))

    def forward(self, x):
        # say (1,2,6)

        b , num_tokens , d_in = x.shape  # where b -> batch , num_tokens is basically rows , and d_in ->cols or features per token
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # .view() is similar to reshape in numpy ,.view() reshapes a tensor without changing the underlying data.
        keys = keys.view(b , num_tokens, self.num_heads , self.head_dim)
        queries = queries.view(b , num_tokens, self.num_heads , self.head_dim)
        values = values.view(b , num_tokens, self.num_heads , self.head_dim)

        # (1 , 2 , 2, 3 )
        keys = keys.transpose(1,2)   # we want tokens to be under their respective heads thats why transpose 1 and 2 position
        queries = queries.transpose(1,2)
        values = values.transpose(1,2) 
       # (1,2,2,3) x (1,2,2,3)
       #  (2,3) @ (3,2) = (2,2) , (1,2,2,2)
        attn_scores = queries @ keys.transpose(2,3)
        mask_bool = self.mask.bool()[:num_tokens , :num_tokens]

        attn_scores.masked_fill_(mask_bool , -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5 , dim = -1)
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1,2) #  (1,2,2,2) @ (1,2,2,3) = (1,2,2,3)  (batch, heads, seq_len, head_dim)
        # .transpose =  (batch, seq_len, heads, head_dim)  we do this so that .view() can correctly merge in further line
        # see because heads x head_dim = d_out , thats why we arrange them back 
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out) 
        # makes sure tensor data is stored continuously in memory in the correct order. , we can also use reshape()
        context_vec = self.out_proj(context_vec) # out_proj is also a learning param now cause the model wants to use heads effectively
        return context_vec
        

In [53]:
batch.shape

torch.Size([2, 6, 3])

In [52]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
